## 1 - загрузка изображения

In [68]:
from show_img import show_img
import cv2
import numpy as np
import math

In [69]:
img = cv2.imread('bibizyana.jpg')
width, height, channels = img.shape
print(f"{height} x {width}")
#show_img(img)
# меняю размер изображения для удобства чтобы влезало на экран
img = cv2.resize(img, (int(height * 0.8), int(width * 0.8)))
width, height, channels = img.shape
print(f"{height} x {width}")
# show_img(img)

1104 x 810
883 x 648


## 2 - евклидовы преобразования

смещение изображения

In [70]:
offset_x = -133
offset_y = 188
offset_matrix = np.float32([[1, 0, offset_x], [0, 1, offset_y]])
offseted_img = cv2.warpAffine(img, offset_matrix, (width, height))
# show_img(offseted_img)

вращение изображения

In [71]:
angle = 33
scale = 1
rotation_matrix = cv2.getRotationMatrix2D(
    center=(width // 2, height // 2), angle=angle, scale=scale)
rotated_img = cv2.warpAffine(offseted_img, rotation_matrix, (width, height))
show_img(rotated_img)

## 3 - ряд преобразований

### афинные преобразования <br>
скосы изображения

In [72]:
curve_matrix_x = np.float32([[1, 0.77, -250], [0, 1, 0]])
curved_x_img = cv2.warpAffine(img, curve_matrix_x, (height, width))
show_img(curved_x_img)

In [73]:
curve_matrix = np.float32([[1, 0.33, -50], [0.33, 1, -100]])
curved_img = cv2.warpAffine(img, curve_matrix, (height, width))
show_img(curved_img)

### проективное преобразование

In [74]:
src_pts = np.float32([[0, 0], [height - 1, 0], [0, width - 1], [height - 1, width - 1]])
dst_pts = np.float32([[0, 0], [int(height * 0.5), 0], [0, int(width * 0.5)], [height - 1, width - 1]])
projective_matrix = cv2.getPerspectiveTransform(src_pts, dst_pts)
img_prj1 = cv2.warpPerspective(img, projective_matrix, (height, width))
show_img(img_prj1)

In [75]:
dst_pts = np.float32([[0, 0], [int(height * 0.87), 0], [0, int(width * 0.2)], [int(height * 0.44), int(width * 0.92)]])
projective_matrix = cv2.getPerspectiveTransform(src_pts, dst_pts)
img_prj2 = cv2.warpPerspective(img, projective_matrix, (height, width))
show_img(img_prj2)

### деформация

In [76]:
img_warp1 = np.zeros(img.shape, dtype=img.dtype)
for i in range(width):
    for j in range(height):
        offset_x = 0
        offset_y = int(23.4 * math.sin(2 * math.pi * j / 123))
        if i + offset_y < height:
            img_warp1[i, j] = img[(i + offset_y) % width, j]
        else:
            img_warp1[i, j] = 0

show_img(img_warp1)

In [77]:
img_warp2 = np.zeros(img.shape, dtype=img.dtype)
for i in range(width):
    for j in range(height):
        offset_x = int(12.3 * math.sin(2 * math.pi * i / 234))
        offset_y = 0
        if j + offset_x < width:
            img_warp2[i, j] = img[i, (j + offset_x) % height]
        else:
            img_warp2[i, j] = 0

show_img(img_warp2)

## 4 - 4 части

In [78]:
img_p1 = img[0: width // 2, 0 : height // 2]
img_p2 = img[0: width // 2, height // 2 : height - 1]
img_p3 = img[width // 2 : width - 1, 0 : height // 2]
img_p4 = img[width // 2 : width - 1, height // 2 : height - 1]

print(img_p1.shape)
print(img_p2.shape)
print(img_p3.shape)
print(img_p4.shape)

(324, 441, 3)
(324, 441, 3)
(323, 441, 3)
(323, 441, 3)


удаление одного из цветового канала

In [79]:
b, g, r = cv2.split(img_p1)
g_empty = np.zeros(g.shape, dtype=g.dtype)
img_empty_chn = cv2.merge([b, g_empty, r])
# show_img(img_empty_chn)

разворот

In [80]:
img_rotated = cv2.rotate(img_p2, cv2.ROTATE_180)
# show_img(img_rotated)

сумма каналов

In [81]:
b3, g3, r3 = cv2.split(img_p3)
b4, g4, r4 = cv2.split(img_p4)
b_new = cv2.add(b3, b4)
g_new = cv2.add(g3, g4)
r_new = cv2.add(r3, r4)
img_combined = cv2.merge([b_new, g_new, r_new])
# show_img(img_combined)

фрагменты из частей

In [82]:
frag_size_x = 100
frag_size_y = 100

img_frag = img_p4.copy()

frag1 = img_p1[0:frag_size_x, 0:frag_size_y]
img_frag[0:frag_size_x, 0:frag_size_y] = frag1

frag2 = img_p2[150:150 + frag_size_x, 150:150 + frag_size_y]
img_frag[150:150 + frag_size_x, 150:150 + frag_size_y] = frag2

frag3 = img_p3[220:320, 340:440]
img_frag[220:320, 340:440] = frag3

# show_img(img_frag)

объединение и рамка

In [ ]:
rescale_size = (441, 323)
img_empty_chn_res = cv2.resize(img_empty_chn, rescale_size)
img_rotated_res = cv2.resize(img_rotated, rescale_size)
#img_combined_res = cv2.resize(img_combined, rescale_size)
#img_frag_res = cv2.resize(img_frag, rescale_size)

top_row = np.hstack((img_empty_chn_res, img_rotated_res))
bottom_row = np.hstack((img_combined, img_frag))

img_result = np.vstack((top_row, bottom_row))

img_result_border = cv2.copyMakeBorder(img_result, 4, 5, 6, 7, cv2.BORDER_CONSTANT, value = [120, 30, 40])

show_img(img_result_border)